# Unit Commitment con Algoritmos Genéticos

Cuaderno listo para **Google Colab** o **JupyterLab**.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
unidades = [
    {'pmax': 50, 'costo_fijo': 20, 'costo_arranque': 30},
    {'pmax': 60, 'costo_fijo': 24, 'costo_arranque': 35},
    {'pmax': 40, 'costo_fijo': 16, 'costo_arranque': 20},
    {'pmax': 30, 'costo_fijo': 12, 'costo_arranque': 15},
]

demanda = [70, 90, 110, 95, 80, 60]

n_unidades = len(unidades)
n_periodos = len(demanda)
longitud_cromosoma = n_unidades * n_periodos

tam_poblacion = 120
n_generaciones = 150
prob_cruce = 0.8
prob_mutacion = 0.05
tam_torneo = 3

random.seed(42)
np.random.seed(42)


In [ ]:
def crear_individuo():
    return np.random.randint(0, 2, size=longitud_cromosoma)

def decodificar(individuo):
    return np.array(individuo).reshape((n_periodos, n_unidades))

def evaluar(individuo):
    matriz = decodificar(individuo)
    costo_total = 0
    deficits = []
    generacion_periodo = []
    estado_anterior = np.zeros(n_unidades, dtype=int)
    for t in range(n_periodos):
        estado = matriz[t]
        generacion = sum(estado[i] * unidades[i]['pmax'] for i in range(n_unidades))
        generacion_periodo.append(generacion)
        deficit = max(0, demanda[t] - generacion)
        deficits.append(deficit)
        costo_total += sum(estado[i] * unidades[i]['costo_fijo'] for i in range(n_unidades))
        for i in range(n_unidades):
            if estado[i] == 1 and estado_anterior[i] == 0:
                costo_total += unidades[i]['costo_arranque']
        costo_total += 200 * deficit
        estado_anterior = estado.copy()
    return costo_total, deficits, generacion_periodo

def fitness(individuo):
    costo_total, _, _ = evaluar(individuo)
    return -costo_total


In [ ]:
def seleccion_torneo(poblacion):
    candidatos = random.sample(poblacion, tam_torneo)
    return max(candidatos, key=fitness).copy()

def cruce_un_punto(padre1, padre2):
    if random.random() < prob_cruce:
        punto = random.randint(1, longitud_cromosoma - 1)
        hijo1 = np.concatenate([padre1[:punto], padre2[punto:]])
        hijo2 = np.concatenate([padre2[:punto], padre1[punto:]])
        return hijo1, hijo2
    return padre1.copy(), padre2.copy()

def mutar(individuo):
    mutado = individuo.copy()
    for i in range(longitud_cromosoma):
        if random.random() < prob_mutacion:
            mutado[i] = 1 - mutado[i]
    return mutado


In [ ]:
poblacion = [crear_individuo() for _ in range(tam_poblacion)]

mejores_costos = []
promedios_costos = []
mejores_deficits = []
mejor_individuo_global = None
mejor_fit_global = -float('inf')

for generacion in range(n_generaciones):
    nueva_poblacion = []
    while len(nueva_poblacion) < tam_poblacion:
        padre1 = seleccion_torneo(poblacion)
        padre2 = seleccion_torneo(poblacion)
        hijo1, hijo2 = cruce_un_punto(padre1, padre2)
        hijo1 = mutar(hijo1)
        hijo2 = mutar(hijo2)
        nueva_poblacion.append(hijo1)
        if len(nueva_poblacion) < tam_poblacion:
            nueva_poblacion.append(hijo2)
    poblacion = nueva_poblacion
    fitness_poblacion = [fitness(ind) for ind in poblacion]
    mejor_generacion = max(poblacion, key=fitness)
    mejor_fit = fitness(mejor_generacion)
    promedio_fit = np.mean(fitness_poblacion)
    costo, deficits, _ = evaluar(mejor_generacion)
    mejores_costos.append(costo)
    promedios_costos.append(-promedio_fit)
    mejores_deficits.append(sum(deficits))
    if mejor_fit > mejor_fit_global:
        mejor_fit_global = mejor_fit
        mejor_individuo_global = mejor_generacion.copy()

costo_final, deficits_final, generacion_final = evaluar(mejor_individuo_global)
matriz_final = decodificar(mejor_individuo_global)
print('Costo total final:', costo_final)
print('Déficit total:', sum(deficits_final))
print('Matriz de compromiso (periodos x unidades):')
print(matriz_final)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mejores_costos, label='Mejor costo')
plt.plot(promedios_costos, label='Costo promedio')
plt.xlabel('Generación')
plt.ylabel('Costo')
plt.title('Evolución del algoritmo genético para Unit Commitment')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, n_periodos + 1), demanda, marker='o', label='Demanda')
plt.plot(range(1, n_periodos + 1), generacion_final, marker='s', label='Generación disponible')
plt.xlabel('Período')
plt.ylabel('Potencia')
plt.title('Demanda vs generación comprometida')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(matriz_final.T, aspect='auto')
plt.xlabel('Período')
plt.ylabel('Unidad')
plt.title('Mapa de encendido/apagado de unidades')
plt.colorbar(label='Estado (0=OFF, 1=ON)')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar([f'P{t+1}' for t in range(n_periodos)], deficits_final)
plt.xlabel('Período')
plt.ylabel('Déficit')
plt.title('Déficit por período')
plt.grid(axis='y')
plt.show()
